In [ ]:
!pip install transformers datasets torch scikit-learn pandas


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import pandas as pd

# train_data.csv ve test_data.csv dosyalarını Drive'dan oku
# Kullanıcı kendi Drive yolunu buraya girecek
DRIVE_PATH = '/content/drive/MyDrive/food-health-predictor/prediction/bert/'

train_df = pd.read_csv(DRIVE_PATH + 'train_data.csv')
test_df = pd.read_csv(DRIVE_PATH + 'test_data.csv')


In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset
import torch

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

class FoodDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

train_dataset = FoodDataset(train_df['text_feature'].values, train_df['label'].values, tokenizer)
test_dataset = FoodDataset(test_df['text_feature'].values, test_df['label'].values, tokenizer)


In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=5
)


In [ ]:
from transformers import Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {'accuracy': accuracy_score(p.label_ids, preds)}

# evaluation_strategy Transformers 4.41'den sonra eval_strategy oldu, eski surumle de uyumlu olsun diye evaluation_strategy kullanilmistir
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_dir='./logs',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

print("Accuracy:", accuracy_score(labels, preds))
print("\nClassification Report:")
print(classification_report(labels, preds, target_names=['A', 'B', 'C', 'D', 'E']))

cm = confusion_matrix(labels, preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['A', 'B', 'C', 'D', 'E'], yticklabels=['A', 'B', 'C', 'D', 'E'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
model.save_pretrained(DRIVE_PATH + 'bert_model')
tokenizer.save_pretrained(DRIVE_PATH + 'bert_model')
print('Model kaydedildi!')
